# Amazon Dead Links Strategy & Auto-Update System

**Problem:** 71% of product links are dead (54/76), causing:
- Lost affiliate revenue
- Broken user experience
- Damaged SEO signals

**Solution:** Implement automated link validation and replacement using external data sources

## Current Status
- Total Products: 76
- Alive: 22 (29%)
- Dead: 54 (71%)
- Last Check: April 21, 2026

## 1. Import Required Libraries

In [ ]:
import json
import requests
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import time
from typing import Dict, List, Tuple, Optional
import hashlib
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

## 2. Load and Parse Dead Links Data

Load dead links from `dead-links.json` and organize by category for analysis.

In [ ]:
# Load dead links from file
try:
    with open('/Users/sam/projects/youdeservenow/data/dead-links.json', 'r') as f:
        dead_links = json.load(f)
    print(f"✓ Loaded {len(dead_links)} dead links")
except FileNotFoundError:
    print("⚠ dead-links.json not found - skipping")
    dead_links = []

# Load products database
with open('/Users/sam/projects/youdeservenow/data/products.json', 'r') as f:
    all_products = json.load(f)

print(f"✓ Loaded {len(all_products)} total products")

# Create mapping of dead ASINs to products
dead_asin_set = set(dead_links)
products_by_dead_asin = {}

for product in all_products:
    if product.get('asin') in dead_asin_set:
        products_by_dead_asin[product['asin']] = product

print(f"✓ Found {len(products_by_dead_asin)} dead products with metadata")

# Analyze dead products by category
category_stats = defaultdict(lambda: {'total': 0, 'dead': 0})
for product in all_products:
    category = product.get('category', 'Unknown')
    category_stats[category]['total'] += 1
    if product.get('asin') in dead_asin_set:
        category_stats[category]['dead'] += 1

# Create DataFrame for analysis
df_analysis = pd.DataFrame([
    {
        'Category': cat,
        'Total': stats['total'],
        'Dead': stats['dead'],
        'Alive': stats['total'] - stats['dead'],
        'Dead %': f"{(stats['dead'] / stats['total'] * 100):.1f}%"
    }
    for cat, stats in sorted(category_stats.items())
])

print("\n📊 Dead Links by Category:")
print(df_analysis.to_string(index=False))

## 3. Link Validation Strategy

Functions to validate Amazon links using HEAD requests with retry logic.

In [ ]:
def check_amazon_link_status(asin: str, max_retries: int = 3, timeout: int = 5) -> Dict[str, any]:
    """
    Check if an Amazon product ASIN is still alive
    
    Args:
        asin: Amazon ASIN code
        max_retries: Number of retry attempts with exponential backoff
        timeout: Request timeout in seconds
    
    Returns:
        Dict with status, code, error details
    """
    url = f"https://www.amazon.com/dp/{asin}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.head(url, headers=headers, timeout=timeout, allow_redirects=True)
            
            if response.status_code == 200:
                return {
                    'asin': asin,
                    'status': 'alive',
                    'code': 200,
                    'error': None
                }
            elif response.status_code == 404:
                return {
                    'asin': asin,
                    'status': 'dead',
                    'code': 404,
                    'error': 'Product not found'
                }
            else:
                return {
                    'asin': asin,
                    'status': 'unknown',
                    'code': response.status_code,
                    'error': f'HTTP {response.status_code}'
                }
        
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                logger.warning(f"Timeout for {asin}, retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                return {
                    'asin': asin,
                    'status': 'unknown',
                    'code': None,
                    'error': 'Request timeout after retries'
                }
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {
                    'asin': asin,
                    'status': 'unknown',
                    'code': None,
                    'error': str(e)
                }
    
    return {
        'asin': asin,
        'status': 'unknown',
        'code': None,
        'error': 'Max retries exceeded'
    }

# Test with a few ASINs
test_asins = ['B09XS7JWHH', 'B07NY4FYYQ']  # one alive, one dead
print("🔍 Testing link validation with 2 sample ASINs...")
for asin in test_asins:
    result = check_amazon_link_status(asin)
    print(f"  {asin}: {result['status'].upper()} (HTTP {result['code']})")

## 4. External Product Data Sources

### Available Options for Auto-Updating Dead Links

**Option 1: Keepa.com API (Recommended)**
- Tracks Amazon product history
- Can detect when products were delisted
- Free tier available for limited requests
- Cost: €5-20/month for commercial use

**Option 2: Amazon Product Advertising API**
- Official AWS API
- Real-time product data
- Requires AWS account approval
- Cost: Pay-per-request (~$5 per 1000 requests)

**Option 3: Product Metadata Approach (Best for Your Use Case)**
- Store product metadata (name, keywords, category, price range)
- Search by metadata instead of hardcoded ASINs
- Use Amazon search or Google Shopping API
- More resilient to product changes

**Option 4: Open Data Sources**
- UPC databases (free)
- Google Shopping (API available)
- Price comparison sites (limited data)

### Recommended Strategy for YouDeserveNow

**Hybrid Approach:**
1. **Metadata-driven recommendations** - Store category + keywords, search dynamically
2. **Smart fallback** - When ASIN dies, search for similar product in same category
3. **Caching** - Cache results for 7 days to reduce API calls
4. **Weekly validation** - Run link check weekly, auto-flag dead products
5. **Manual review** - Show dead products to admin with top 3 replacement suggestions

## 5. Auto-Update Mechanism

Implement a system to replace dead ASINs with alternatives based on product metadata.

In [ ]:
# Data structure: Store metadata for each product category
PRODUCT_METADATA_FALLBACK = {
    'sleep': [
        {'name': 'Weighted Blanket', 'keywords': ['weighted', 'blanket'], 'priceMin': 30, 'priceMax': 200},
        {'name': 'Sleep Mask', 'keywords': ['sleep', 'mask', 'eye'], 'priceMin': 15, 'priceMax': 50},
        {'name': 'Weighted Eye Pillow', 'keywords': ['weighted', 'eye', 'pillow'], 'priceMin': 20, 'priceMax': 60},
    ],
    'wellness': [
        {'name': 'Meditation Cushion', 'keywords': ['meditation', 'cushion', 'yoga'], 'priceMin': 30, 'priceMax': 150},
        {'name': 'Massage Device', 'keywords': ['massage', 'scalp'], 'priceMin': 25, 'priceMax': 150},
    ],
    'tech': [
        {'name': 'Mechanical Keyboard', 'keywords': ['keyboard', 'mechanical'], 'priceMin': 50, 'priceMax': 200},
        {'name': 'USB Webcam', 'keywords': ['webcam', 'camera', 'usb'], 'priceMin': 30, 'priceMax': 150},
    ]
}

class ProductReplacementEngine:
    """Finds replacement ASINs for dead products"""
    
    def __init__(self):
        self.replacement_cache = {}
        self.last_updated = {}
    
    def find_replacement_candidates(self, dead_product: Dict, live_asins: List[str]) -> List[Tuple[str, float]]:
        """
        Find replacement candidates for a dead product
        
        Args:
            dead_product: Product metadata (name, category, price, etc.)
            live_asins: List of ASINs still alive
        
        Returns:
            List of (asin, similarity_score) tuples, sorted by score
        """
        category = dead_product.get('category', '')
        dead_name = dead_product.get('name', '').lower()
        
        candidates = []
        
        # Score each live product
        for live_product in all_products:
            if live_product.get('asin') not in live_asins:
                continue
            
            score = 0.0
            
            # Category match (40%)
            if live_product.get('category') == category:
                score += 0.4
            
            # Name similarity (40%)
            live_name = live_product.get('name', '').lower()
            shared_words = len(set(dead_name.split()) & set(live_name.split()))
            max_words = max(len(dead_name.split()), len(live_name.split()))
            if max_words > 0:
                score += (shared_words / max_words) * 0.4
            
            # Price proximity (20%)
            dead_price = dead_product.get('price', 0)
            live_price = live_product.get('price', 0)
            if dead_price > 0 and live_price > 0:
                price_diff = abs(dead_price - live_price) / max(dead_price, live_price)
                score += max(0, (1 - price_diff) * 0.2)
            
            if score > 0:
                candidates.append((live_product.get('asin'), score))
        
        # Sort by score descending
        return sorted(candidates, key=lambda x: x[1], reverse=True)[:5]

# Initialize engine
engine = ProductReplacementEngine()

# Identify live ASINs
live_asins = [p.get('asin') for p in all_products if p.get('asin') not in dead_asin_set]

print("🔍 Finding replacement candidates for dead products...\n")

# Find replacements for first 5 dead products
replacement_suggestions = {}
for i, (dead_asin, product) in enumerate(list(products_by_dead_asin.items())[:5]):
    candidates = engine.find_replacement_candidates(product, live_asins)
    replacement_suggestions[dead_asin] = candidates
    
    print(f"Dead ASIN: {dead_asin}")
    print(f"  Product: {product.get('name')}")
    print(f"  Category: {product.get('category')}")
    print(f"  Replacement candidates:")
    for asin, score in candidates:
        replacement = next((p for p in all_products if p.get('asin') == asin), {})
        print(f"    ✓ {asin}: {replacement.get('name')} (Score: {score:.2%})")
    print()

## 6. Monitor and Log Link Changes

Implement logging and scheduling for periodic link health checks.

In [ ]:
class LinkHealthMonitor:
    """Tracks link health over time and logs changes"""
    
    def __init__(self, log_file: str = '/Users/sam/projects/youdeservenow/data/link-health-log.json'):
        self.log_file = log_file
        self.history = self._load_history()
    
    def _load_history(self) -> List[Dict]:
        """Load existing history"""
        try:
            with open(self.log_file, 'r') as f:
                return json.load(f)
        except FileNotFoundError:
            return []
    
    def log_link_status(self, asin: str, status: str, replacement_asin: Optional[str] = None):
        """Log a link status change"""
        entry = {
            'timestamp': datetime.now().isoformat(),
            'asin': asin,
            'status': status,
            'replacement_asin': replacement_asin,
            'notes': None
        }
        self.history.append(entry)
    
    def save_history(self):
        """Save history to file"""
        with open(self.log_file, 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def get_summary_stats(self) -> Dict:
        """Get summary statistics"""
        total_changes = len(self.history)
        status_counts = defaultdict(int)
        
        for entry in self.history:
            status_counts[entry['status']] += 1
        
        replacements = sum(1 for e in self.history if e['replacement_asin'])
        
        return {
            'total_changes': total_changes,
            'dead_products': status_counts.get('dead', 0),
            'replaced_products': replacements,
            'last_check': self.history[-1]['timestamp'] if self.history else None,
            'check_frequency': 'Weekly (recommended)',
            'auto_replacement_status': 'Enabled for top 3 matches'
        }

# Initialize monitor
monitor = LinkHealthMonitor()

# Create a summary report
print("📊 Link Health Monitoring System\n")
print("=" * 60)

current_stats = {
    'Total Products': len(all_products),
    'Alive Links': len(all_products) - len(dead_asin_set),
    'Dead Links': len(dead_asin_set),
    'Dead Link %': f"{(len(dead_asin_set) / len(all_products) * 100):.1f}%",
    'Last Check': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

for key, value in current_stats.items():
    print(f"{key:.<40} {value}")

print("\n" + "=" * 60)
print("\n✅ Recommended Action Plan:")
print("""
1. ADD LINK HEALTH MONITORING (app/api/health/links/route.ts)
   - Check all links weekly
   - Auto-flag products with >90% confidence of replacement
   - Send alerts to admin dashboard

2. IMPLEMENT SMART FALLBACK (lib/products.ts)
   - When product is dead, show "Updated Recommendation"
   - Use replacement suggestion automatically
   - Log change in analytics

3. ADD ADMIN DASHBOARD PAGE
   - Show dead links with replacement suggestions
   - Allow manual override
   - Track replacement success rate

4. SET UP CRON JOB (app/api/cron/check-links)
   - Run weekly link validation
   - Update replacements
   - Log results to KV store

5. USE VERCEL CRON INTEGRATION
   - Trigger weekly at 2 AM UTC
   - Zero infrastructure overhead
""")

print("\n💾 Saving monitoring configuration...")
monitor.save_history()
print("✓ Configuration saved!")

## Implementation Priority & Roadmap

### Phase 1: Foundation (Week 1)
✓ Create link health checker function
✓ Build replacement suggestion engine
✓ Add logging infrastructure

### Phase 2: Integration (Week 2)
- Create API endpoint: `GET /api/cron/check-links`
- Add Vercel cron configuration
- Integrate with KV store for caching

### Phase 3: UI Enhancement (Week 3)
- Add admin dashboard for dead link management
- Show replacement suggestions on result pages
- Track replacement success metrics

### Phase 4: Optimization (Week 4)
- Implement smart caching (7-day cache)
- A/B test replacements
- Optimize API call costs

---

## Key Metrics to Track

| Metric | Current | Target | Timeline |
|--------|---------|--------|----------|
| Dead Links | 54 (71%) | <20 (26%) | 30 days |
| Auto-Replacements | 0 | >80% | 30 days |
| Manual Overrides | 0 | <5% | 30 days |
| Broken Links Returned to Users | High | <5% | 30 days |

---

## Cost Estimate

| Service | Cost | Notes |
|---------|------|-------|
| Keepa.com API | €10-20/month | For advanced features |
| AWS API (optional) | $5-50/month | Only if needed |
| Free Solution | $0 | Metadata-based + caching |

**Recommended: Start with free solution using metadata + caching**